# NB00 – Business Case

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Mitglieder:** Patrik Neunteufel · Senthuran Elankeswaran · Cyril Saladin | **Datum:** März 2026

---

*Konsolidierter Bericht: Wirtschaftlichkeit, Segmentanalyse und Empfehlungen.*


| [← NB04 Visualisierungen](04_Visualisierungen.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB01 Daten Laden →](01_Daten_Laden.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_NB_00'></a>

[Einleitung](#einleitung_NB_00)  
[Initialisierung](#initialisierung_NB_00)  
1 [Daten](#daten_NB_00)  
2 [Methodik](#methodik_NB_00)  
3 [Ergebnisse](#ergebnisse_NB_00)  
[Fazit und Empfehlungen](#fazit-und-empfehlungen_NB_00)  
[Kür-Erweiterungen](#kuer-erweiterungen_NB_00)  
[Abschluss](#abschluss_NB_00)  


## Einleitung <a id='einleitung_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

### Geschäftliches Problem

Die Schweiz steht vor einer doppelten Herausforderung: Einerseits soll der Anteil erneuerbarer Energien stark ausgebaut werden, andererseits müssen gleichzeitig die Netzstabilität und die Versorgungssicherheit gewährleistet bleiben. Erneuerbare Energien wie Photovoltaik und Wind produzieren Strom unregelmässig und verstärken die Schwankungen im Netz.

**Die Kernfrage dieses Projekts:** Können private und industrielle Batteriespeicher, die **unabhängig von einer Solaranlage** betrieben werden, durch gezieltes Laden bei Niedrigpreisen und Einspeisen bei Hochpreisen gleichzeitig wirtschaftlich rentabel sein **und** zur [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung) beitragen?

Dieser Mechanismus wird als **Grid-Arbitrage** bezeichnet.

### Wer ist interessiert?

| Stakeholder | Interesse |
|-------------|-----------|
| Privathaushalte | Kostenersparnis durch günstigeres Laden, Zusatzeinnahmen |
| Gewerbebetriebe | Lastspitzenvermeidung + Arbitrage-Erlöse |
| Netzbetreiber (Swissgrid) | Weniger Netzausbaubedarf durch aggregierte Speicher |
| Politik / [BFE](../organisation/O_02_Glossar.ipynb#g-bfe) | Datenbasis für Förderstrategien, Energiestrategie 2050 |
| Investoren | Portfolio-Optimierung mit Stromspeicher-Projekten |


## Initialisierung<a id='initialisierung_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

Lädt `../sync/config.json`, definiert `show_chart()` zum Einbetten der erzeugten Charts, und liest Ergebnisse aus `../sync/transfer.json`.

*Charts werden aus `charts/` geladen (erzeugt durch NB3). Einzelplots sind grösser und direkt berichtstauglich.*


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
from IPython.display import display, Image

# Versionen anzeigen für Reproduzierbarkeit
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Setup – Konfiguration & Chart-Anzeige:**  
Lädt `../sync/config.json`, definiert `show_chart()` zum Einbetten der erzeugten Charts aus `../output/charts/`, und liest Kennzahlen aus `../sync/transfer.json`.

In [ ]:
# ── Konfiguration laden (Single Source of Truth) ──────────────────────────────
if os.path.exists('../sync/config.json'):
    with open('../sync/config.json') as _f:
        CFG = json.load(_f)
else:
    CFG = {}
    print('⚠️  ../sync/config.json nicht gefunden')

FORCE_RELOAD  = CFG.get('force_reload', {})
SZ_AKTIV      = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_INTER     = os.path.join('../data', 'intermediate')
DIR_INTER_SZ  = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR    = os.path.join('../output', 'charts', SZ_AKTIV)

print(f'../sync/config.json geladen | Szenario: {SZ_AKTIV}')
print(f'Charts-Verzeichnis : {os.path.abspath(CHARTS_DIR)}')

# ── Chart-Anzeige-Funktion ────────────────────────────────────────────────────
def show_chart(filename, caption='', width=950):
    """Zeigt einen erzeugten Chart aus output/charts/<szenario>/."""
    path = os.path.join(CHARTS_DIR, filename)
    if not os.path.exists(path):
        print(f'❌  Nicht vorhanden: {path}')
        print(   '   → NB04 Visualisierungen zuerst ausführen.')
        return
    display(Image(filename=path, width=width))
    if caption:
        print(f'\n{caption}\n')

# ── Szenarien-Metadaten laden (_gz_mode für Zelle 4.4) ───────────────────────
SZ_FILE  = os.path.join(DIR_INTER_SZ, 'netzentlastung_szenarien.csv')
_gz_mode = 'unbekannt'
_gz_rate = '?'
if os.path.exists(SZ_FILE):
    _df_sz = pd.read_csv(SZ_FILE)
    if 'gleichzeitigkeit' in _df_sz.columns:
        _gz_mode = _df_sz['gleichzeitigkeit'].iloc[0]
        _gz_rate = f"{_df_sz['rate_pct'].iloc[0]:.0f}%"
        print(f'Szenarien geladen — Gleichzeitigkeit: {_gz_mode} ({_gz_rate})')
    else:
        print('Szenarien geladen (Format ohne Gleichzeitigkeit-Spalte)')
else:
    print('⚠️  netzentlastung_szenarien.csv nicht gefunden → NB03 zuerst ausführen')

charts = (sorted(f for f in os.listdir(CHARTS_DIR) if f.endswith('.png'))
          if os.path.exists(CHARTS_DIR) else [])
print(f'{len(charts)} Chart(s) verfügbar in {CHARTS_DIR}')

# ── Transfer-Daten laden (Kennzahlen aus NB03) ────────────────────────────────
_tf_path = '../sync/transfer.json'
if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0:
    TF         = json.load(open(_tf_path))
    _dt        = TF.get('datenzeitraum', {})
    _sim       = TF.get('simulation', {})
    TF_N_YEARS = _dt.get('n_years', None)
    TF_START   = _dt.get('start_date', 'unbekannt')
    TF_END     = _dt.get('end_date', 'unbekannt')
    TF_SPREAD  = _sim.get('spread_mean_eur_mwh', None)
    TF_ECON    = _sim.get('wirtschaftlichkeit', {})
    TF_HYB     = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    TF_KUER    = CFG.get('kuer_aktiv', {})
    print(f'../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh')
else:
    TF = {}; TF_N_YEARS = None; TF_START = TF_END = 'unbekannt'
    TF_SPREAD = None; TF_ECON = {}; TF_HYB = {}; TF_KUER = {}
    print('⚠️  ../sync/transfer.json nicht gefunden — NB01–NB03 zuerst ausführen')


In [ ]:
# ── ⚙ Markdown-Prüfwerte ──────────────────────────────────────────────────────
# Diese Werte erscheinen als ⚙ im Markdown-Text.
# Nach jeder config-Änderung: Ausgabe mit ⚙-Stellen im Text vergleichen!
_w = CFG['pflicht']['wirtschaftlichkeit']
_s = CFG['pflicht']['simulation']
_lz = CFG['pflicht'].get('langzeit', {})
_lt = _w['lifetime_j']
print('=== ⚙ MARKDOWN-PRÜFWERTE (config-abhängig) ===')
print(f'  ZIEL_ROI     = {round(100/_lt,2)}%  (= 100 / {_lt} Jahre)')
print(f'  LIFETIME     = {_lt}J  |  LIFETIME_LONG = {_lz.get("lifetime_long_j","?")}J')
print(f'  EFFICIENCY   = {_s["efficiency_roundtrip"]*100:.0f}%')
print(f'  SOC          = {_s["soc_min_pct"]*100:.0f}%–{_s["soc_max_pct"]*100:.0f}%')
print(f'  OPEX_RATE    = {_w["opex_rate"]*100:.1f}%')
print(f'  TRIGGER_EUR  = {CFG.get("monitoring",{}).get("spread_trigger_eur_mwh",30)} EUR/MWh')


## 1. Daten <a id='daten_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

### Verwendete Datensätze

**Datensatz 1: [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Day-Ahead Spot-Preise (Schweiz)**  
Stündliche Day-Ahead Preise für die CH-Bietungszone (`10YCH-SWISSGRIDZ`) in EUR/MWh, 2023–2024. Quelle: [transparency.entsoe.eu](https://transparency.entsoe.eu)

**Datensatz 2: ENTSO-E Netzlast CH**  
Stündliche Systemlast des Schweizer Regelblocks [MW], gleiche Quelle und Zeitraum.


## 2. Methodik <a id='methodik_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

### Dispatch-Modell

Die Batterie-[Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation verwendet ein tagesbasiertes Schwellenwertmodell:
- **Laden:** Wenn der aktuelle Preis unter dem 25. Perzentil des jeweiligen Tages liegt und der Ladezustand unter 95%⚙ ist
- **Einspeisen:** Wenn der aktuelle Preis über dem 75. Perzentil des Tages liegt und der Ladezustand über 5%⚙ ist
- **Idle:** Sonst — keine Aktion. Dies tritt auf, wenn der tägliche Spread zu klein ist, um die Rundlaufverluste zu decken: Dispatch lohnt sich nur wenn `p75 × η > p25` ([Dispatch-Break-even](../organisation/O_02_Glossar.ipynb#g-dispatch-breakeven)).

Der [Round-Trip-Wirkungsgrad](../organisation/O_02_Glossar.ipynb#g-rte) beträgt 92%⚙ (realistischer Wert für Lithium-Ionen-Batterien).

### Simulierte Segmente

| Segment | Kapazität | Leistung | Stückzahl CH (Schätzung) |
|---------|-----------|----------|--------------------------|
| Privat | 10 kWh | 5 kW | 200'000 |
| Gewerbe | 100 kWh | 30 kW | 10'000 |
| Industrie | 1 MWh | 200 kW | 1'000 |
| Utility | 10 MWh | 1'000 kW | 100 |

### Wirtschaftlichkeitsrechnung

[CAPEX](../organisation/O_02_Glossar.ipynb#g-capex)-Annahmen (2024): Privat 400 EUR/kWh⚙, Gewerbe 300 EUR/kWh⚙, Industrie 220 EUR/kWh⚙, Utility 180 EUR/kWh⚙. Betriebskosten 1.5%⚙ des CAPEX pro Jahr. Nutzungsdauer 12 Jahre⚙.


## 3. Ergebnisse <a id='ergebnisse_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

### 4.1 Spot-Preis-Struktur

Das Tages- und Saisonalprofil der Schweizer Day-Ahead Preise zeigt zwei klare Muster: **Tief im solaren Mittagsfenster (Frühjahr/Sommer)** und bei niedrigem Verbrauch (Nachtstunden), **hoch bei Morgen- und Abendspitze**. Die Heatmap zeigt den tiefsten Durchschnittspreis im Mittag der Sommermonate (Solar-Erzeugungsmaximum) und die höchste [Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet) in den Übergangsjahreszeiten.


In [ ]:
show_chart('nb04_heatmap_preis.png',
           'Chart 2: Ø Spot-Preis CH — Stunde × Monat 2023–2024', width=900)

**Interpretation Heatmap:** Die Farbcodierung zeigt das Arbitrage-Potenzial direkt: **Grüne Felder** (günstige Stunden) sind das optimale Ladefenster, **dunkle/rote Felder** das Einspeisefenster. Zwei Muster stechen hervor:

- **Sommer-Mittag (10–14 Uhr, Juni–August):** Tiefste Durchschnittspreise des Jahres durch Solar-Überproduktion — ideales Ladefenster. Im Extremfall entstehen hier Negativpreise.
- **Abendspitze (17–21 Uhr, Okt–Feb):** Höchste Durchschnittspreise durch gleichzeitige Heiz- und Berufslast — ideales Einspeisefenster.

Der **Frühling (März–Mai)** zeigt dabei die grösste Preisspanne zwischen Mittag und Abend — die [Duck Curve](../organisation/O_02_Glossar.ipynb#g-duck-curve) ist in dieser Jahreszeit am ausgeprägtesten.

### 4.2 Tagesprofil: Netzlast & Arbitrage-Fenster

Chart 3 zeigt die zeitliche Überlagerung von Netzlast und Preis: Das **Ladefenster (blau, ~09–13 Uhr)** fällt ins Solar-Mittagstief, wenn Preise und Netzlast tief sind. Das **Einspeisefenster (rot, ~15–20 Uhr)** trifft die Abendspitze — genau dann ist die Netzlast am höchsten und der Beitrag zur [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung) am grössten.


In [ ]:
show_chart('nb04_tagesprofil_einzel.png',
           'Chart 3: Ø Tagesprofil Netzlast & Spot-Preis mit Arbitrage-Fenstern', width=900)

### 4.3 Wirtschaftlichkeit

Die folgenden Charts zeigen die Wirtschaftlichkeit aus vier Winkeln. Alle Zahlen basieren **ausschliesslich auf Arbitrage-Erlösen** — Regelenergiemarkt (FCR/aFRR), Peak-Shaving-Prämien und [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking) sind nicht eingerechnet.

**Kernergebnis:** Kein Segment erreicht den Ziel-[ROI](../organisation/O_02_Glossar.ipynb#g-roi) von 8.33%⚙ p.a. bei reiner Arbitrage. Industrie liegt mit 2.3%📊 am nächsten, Privat mit 0.6%📊 am weitesten entfernt. Keines der Segmente erreicht den Break-Even innerhalb der modellierten 12 Jahre⚙ — auch auf 20 Jahre⚙ (Chart 1e) bleibt der kumulierte Cashflow negativ.


In [ ]:
show_chart('nb04_amortisation.png',
           'Amortisationskurven: kumulierter Cashflow über 12 Jahre', width=900)

**Interpretation Amortisation:** Alle vier Kurven starten im negativen Bereich (CAPEX-Auszahlung) und steigen linear an. Keine Kurve schneidet die Nulllinie innerhalb von 12 Jahren — kein Segment erreicht den Break-Even in der modellierten Lebensdauer. Utility startet am tiefsten (−1'800 kEUR) und steigt am steilsten, bleibt aber auch nach 12J noch tief im negativen Bereich. Die [symlog](../organisation/O_02_Glossar.ipynb#g-symlog)-Y-Achse ermöglicht, alle vier Grössenordnungen (4 kEUR bis 1'800 kEUR CAPEX) gleichzeitig sichtbar zu machen.

In [ ]:
show_chart('nb04_roi.png',
           'ROI pro Jahr nach Segment (nur Arbitrage-Erlöse)', width=900)

**Interpretation ROI:** Die gestrichelte Linie markiert den **Break-Even-ROI von 8.33%⚙/Jahr** (= Lebensdauer 12J⚙ amortisiert). Kein Segment erreicht ihn: Industrie kommt mit **4.5%📊** am nächsten, Privat bleibt bei **1.7%📊**. Der scheinbare Paradox — Utility hat höchsten Absoluterlös aber nur 2.9% ROI — erklärt sich durch den proportional noch höheren CAPEX (180 EUR/kWh × 10'000 kWh = 1.8 Mio EUR). Der ROI-Chart zeigt klar: Das Problem ist nicht die Technik, sondern das **Verhältnis von Spread-Niveau zu CAPEX-Niveau** unter aktuellen CH-Marktbedingungen.

In [ ]:
show_chart('nb04_erloese_kwh.png',
           'Erlös pro kWh Kapazität — normierter Segmentvergleich', width=900)

**Erlös pro kWh (normiert):** Dieser Chart macht Segmente grössenunabhängig vergleichbar — alle Segmente erzielen ~13 EUR pro installierter kWh Kapazität über ~2 Jahre. Ausnahme: **Utility (7.9 EUR/kWh📊)** schneidet schlechter ab. Der Grund liegt in der [C-Rate](../organisation/O_02_Glossar.ipynb#g-c-rate): Bei 10 MWh und 1 MW Konverterleistung (C-Rate 0.1C) kann die Batterie in einem typischen 2-Stunden-Preisspread-Fenster nur 20% ihrer Kapazität nutzen. Gewerbe (13.7 EUR/kWh📊) mit C-Rate 0.3C nutzt das Preisspread-Fenster am effizientesten.

In [ ]:
show_chart('nb04_capex_ertrag.png',
           'CAPEX vs. kumulierter Netto-Erlös (12 Jahre)', width=900)

**[CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) vs. kumulierter Netto-Erlös (12J):** Dieser Chart zeigt das strukturelle Kernproblem in absoluten Zahlen. Bei Utility stehen **1'800 kEUR CAPEX** einer kumulierten Netto-Einnahme von ~**625 kEUR** (12J📊) gegenüber — die Investition wird zu ~35% zurückverdient. Industrie steht am besten: **220 kEUR CAPEX** vs. ~**118 kEUR** Netto über 12J (~54% Rückfluss). Für vollständige Rentabilität bräuchte es entweder einen 2–3× höheren Spread oder ~60% tiefere CAPEX — oder beides kombiniert mit [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking).

### 4.4 Netzentlastungs-Potenzial

Aggregierte Batterien können die Schweizer Spitzenlast (Basis: 10.5 GW) messbar reduzieren. Die Wirkung steigt stark mit dem [Rollout-Szenario](../organisation/O_02_Glossar.ipynb#g-rollout-szenario): Von 1.3%📊 (Moderat, 2027) bis 20.2%📊 (Transformativ, 2035).

> **Gleichzeitigkeits-Annahme:** Die Ergebnisse basieren auf dem aktiven Szenario in `../sync/config.json → szenarien.gleichzeitigkeit_aktiv`. Zum Umschalten: Wert ändern → NB02 + NB03 neu ausführen → Charts aktualisieren sich automatisch.


In [ ]:
show_chart('nb04_spitzenlast.png',
           'Resultierende Spitzenlast je Szenario [GW]', width=800)

In [ ]:
# Gleichzeitigkeit-Info aus den geladenen Szenarien
if _gz_mode != 'unbekannt':
    print(f'Gleichzeitigkeit (aus NB2): {_gz_mode} ({_gz_rate})')
    if _gz_mode == 'realistisch':
        print('  → 40% der Batterien dispatchen gleichzeitig (unkoordinierter Markt)')
        print('  → Zum Vergleich: optimistisch = 70% (VPP-koordiniert) ergibt ~1.75× höhere Entlastung')
    else:
        print('  → 70% der Batterien dispatchen gleichzeitig (VPP-koordinierter Dispatch)')
        print('  → Zum Vergleich: realistisch = 40% (unkoordiniert) ergibt ~0.57× der Entlastung')


In [ ]:
show_chart('nb04_spitzenlast_reduktion.png',
           'Spitzenlast-Reduktion [%] je Szenario', width=800)

**Interpretation [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung):** Die Szenarien zeigen einen exponentiellen Effekt: Erst ab dem **[Transformativ-Szenario](../organisation/O_02_Glossar.ipynb#g-rollout-szenario)** (2035: 800'000 Privat, 2'000 Industrie-Einheiten) wird die Systemwirkung für Swissgrid wirklich relevant — 20.2%📊 Spitzenlastreduzierung ist eine Grösse, die Netzausbauinvestitionen verschiebt oder verhindert. Das [Moderat-Szenario](../organisation/O_02_Glossar.ipynb#g-rollout-szenario) (2027) bleibt mit 1.3%📊 systemisch kaum spürbar. Entscheidend ist die **Gleichzeitigkeitsannahme** (40%⚙ realistisch): Nicht alle Batterien entladen gleichzeitig — koordinierter [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-Dispatch würde bis zu 70%⚙ erreichen und die Kurven entsprechend steiler machen.

### 4.5 Saisonale Rentabilität

Der Arbitrage-Spread variiert stark saisonal — entgegen der Erwartung ist **Frühling** die renditestärkste Saison, nicht Winter. Die Kombination aus noch hoher Heizlast und früher Solarproduktion erzeugt den grössten Intra-Tag-Preisunterschied.

> ⚠️ **Begriffsklarstellung:** Diese Tabelle verwendet zwei verschiedene Spread-Grössen — sie sind **nicht direkt vergleichbar**:
> - **Marktpreisspanne (max−min):** Täglicher Höchstpreis minus Tiefstwert. Theoretische Obergrenze; setzt perfektes Timing *und* unbegrenzte Konverterleistung voraus (real limitiert durch C-Rate — z.B. 5 kW / 10 kWh = C-Rate 0.5 → max. 50% der Kapazität pro Stunde nutzbar).
> - **Simulierter Monatserlös (p75−[p25](../organisation/O_02_Glossar.ipynb#g-p25p75)-Basis):** Tatsächlich erzielbarer Erlös aus der [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation pro MWh Kapazität und Monat. Basis für den Rentabilitäts-Trigger (> 30 EUR/MWh⚙, vgl. K_03).

| Saison | Ø Marktpreisspanne (max−min) | Simulierter Monatserlös (Chart 5b) | Charakteristik |
|--------|------------------------------|-------------------------------------|----------------|
| **Frühling** | ~139 EUR/MWh📊 | Mai: ~38 EUR/MWh·Monat📊 (Maximum) | Höchste Arbitrage-Rendite |
| Sommer | ~122 EUR/MWh📊 | Jul–Aug: ~22–25 EUR/MWh·Monat📊 | Viele Negativpreise (Ladechancen) |
| Winter | ~85 EUR/MWh📊 | Jan–Mar: ~19–20 EUR/MWh·Monat📊 | Gleichmässiger, geringerer Spread |
| Herbst | ~81 EUR/MWh📊 | Sep–Okt: ~21–26 EUR/MWh·Monat📊 | Übergangsphase |

**Negativpreise** treten hauptsächlich im Frühjahr/Sommer auf (Apr–Jun) — dort kann die Batterie kostenlos laden und erhöht damit den effektiven Erlös. Im Winter sind Negativpreise selten.


In [ ]:
show_chart('nb04_saisonal_frühling.png',
           'Frühling-Tagesprofil: Höchster Spread (139 EUR/MWh) — optimale Arbitrage-Bedingungen', width=900)

**Frühling (März–Mai):** Die renditestärkste Saison mit ~139 EUR/MWh Marktpreisspanne (max−min) und simulierten Monatserlösen bis ~38 EUR/MWh im Mai. Der Grund ist der **[Duck-Curve-Effekt](../organisation/O_02_Glossar.ipynb#g-duck-curve)** in seiner stärksten Form: Solar ist bereits aktiv (Mittagstief), aber die Heizlast ist noch hoch (Abendspitze) — die Preisspanne zwischen Tief und Hoch ist maximal. Der Frühlings-Abend (19:00 Uhr) ist der rentabelste Einspeisezeitpunkt des ganzen Jahres.

In [ ]:
show_chart('nb04_saisonal_sommer.png',
           'Sommer-Tagesprofil: Solar-Mittagstief erzeugt Negativpreise — kostenlose Ladezyklen', width=900)

**Sommer (Juni–August):** Das Solar-Mittagstief (10–14 Uhr) erzeugt zeitweise **Negativpreise** (~17 Stunden/Jahr📊 im Schnitt) — die Batterie lädt in diesen Stunden gratis und wird sogar für die Aufnahme vergütet. Das Ladefenster verschiebt sich auf den Mittag statt in die Nacht. Der effektive Erlös steigt dadurch trotz moderatem Tagesspread: Die Ladekostenersparnis bei Negativpreisen ist ein echter Bonus. Sommer-Arbitrage erfordert einen saisonal adaptierten [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Algorithmus — ein statisches Nacht-Ladefenster verpasst den Grossteil des Potenzials.

In [ ]:
show_chart('nb04_saisonal_winter.png',
           'Winter-Tagesprofil: Geringster Spread (85 EUR/MWh) — Kontrastbeispiel zur renditeschwächsten Saison', width=900)

**Winter (Dezember–Februar):** Mit ~85 EUR/MWh Marktpreisspanne (max−min) ist der Winter die renditeschwächste Saison — entgegen der intuitiven Erwartung. Die Nachfrage ist zwar hoch (Heizung), aber das Angebot ist gleichmässiger, weil Pumpspeicher regulär einspeisen und keine extremen Solar-Mittagstäler entstehen. Der simulierte Monatserlös liegt bei ~19–20 EUR/MWh, das Ladefenster verschiebt sich auf die frühen Morgenstunden (02–06 Uhr). Winter zeigt, dass Preisniveau allein nicht reicht — es braucht **zeitliche Preisvolatilität**, nicht nur hohes Preisniveau.

In [ ]:
show_chart('nb04_saisonal_roi.png',
           'Monatlicher Arbitrage-Spread & Negativpreis-Stunden', width=900)

---

### 4.6 Grenzen der reinen Arbitrage

Die Ergebnisse in 4.3–4.5 beschreiben **reine Grid-Arbitrage** unter aktuellen CH-Marktbedingungen. Mehrere Parameter können den Business Case wesentlich verbessern — sie sind nicht Gegenstand dieses Pflicht-Notebooks, werden aber in den Kür-Notebooks systematisch untersucht:

- **Steigender Spread / sinkende CAPEX** → verbessert [ROI](../organisation/O_02_Glossar.ipynb#g-roi) direkt
- **Erlösstacking** (FCR/aFRR/VPP) → Erlösquellen neben Arbitrage
- **DA-optimaler Dispatch** → höhere Ausnutzung der Preisspitzen
- **Standortwahl nach BVI** → Netzentlastungswert steigt bei richtiger Zone
- **Andere Technologien & Eigenverbrauch** → andere Kostenstruktur, andere Amortisationslogik

> Die strategische Synthese aller Faktoren — unter welchen Bedingungen [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) wirtschaftlich sinnvoll wird — liefert **[K_00 – Business Strategy](../kuer/K_00_Business_Strategy.ipynb)** (Kür-Notebook, Ordner `kuer/`).


## Fazit und Empfehlungen <a id='fazit-und-empfehlungen_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

**Grid-Arbitrage ist technisch realisierbar, bei aktuellen CH Marktpreisen aber noch nicht rentabel.** Kein Segment erreicht den Ziel-[ROI](../organisation/O_02_Glossar.ipynb#g-roi) von 8.33%⚙ p.a. (= 100 / 12 Jahre⚙ Lebensdauer) durch reine Arbitrage:

| Segment | ROI (Arbitrage) | Break-Even | Bewertung |
|---------|----------------|------------|-----------|
| Privat 10 kWh | 1.7%📊 | > 20 Jahre📊 | Nicht rentabel |
| Gewerbe 100 kWh | 3.1%📊 | > 20 Jahre📊 | Nicht rentabel |
| Industrie 1 MWh | 4.5%📊 | > 20 Jahre📊 | Nächste am Ziel |
| Utility 10 MWh | 2.9%📊 | > 20 Jahre📊 | Nicht rentabel |

**Hauptursache:** Der Schweizer Day-Ahead Spread (~20–38 EUR/MWh je Monat) ist zu gering um [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) von 180–400 EUR/kWh⚙ in der modellierten Lebensdauer zurückzuverdienen.

**Wann kippt der Business Case?** Mehrere Parameter können den Business Case wesentlich verbessern (Abschnitt 4.6): steigender Spread, sinkende CAPEX, [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking), DA-optimaler [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) und standortbasierter Netzwert. Eine vollständige Szenarienanalyse über alle diese Faktoren liefert **[K_00 – Business Strategy](../kuer/K_00_Business_Strategy.ipynb)** — das Kür-Notebook das alle Einzelanalysen aus `kuer/` zusammenführt.

> Weiterführende strategische Empfehlungen und räumliche Optimierung: → [K_00 – Business Strategy](../kuer/K_00_Business_Strategy.ipynb)

---
*Daten: [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Transparency Platform (DL-DE-BY-2.0) · ENTSO-E query\_load (Netzlast).*  
*Kür-Analysen (K_00–K_11 in `kuer/`): Business Strategy, Räumliche Analyse, Cross-Border, Marktdynamik, Animationen, [Revenue Stacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking), Dispatch, Technologievergleich, Alternative Speicher, Eigenverbrauch, Produktsteckbrief, Kombinierte Simulation.*


---

## Kür-Erweiterungen <a id='kuer-erweiterungen_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

Die folgenden Notebooks vertiefen den Business Case um weitere Analysedimensionen. Sie liegen im Ordner **`kuer/`** und setzen die Pflicht-Notebooks (`notebooks/NB01–NB04`) voraus — Daten und Zwischenergebnisse fliessen via `../sync/transfer.json` oder `../data/`.

**Einstiegspunkt:** [K_00 – Business Strategy](../kuer/K_00_Business_Strategy.ipynb) fasst alle Kür-Analysen strategisch zusammen.

| Notebook | Inhalt |
|----------|--------|
| [K_00 – Business Strategy](../kuer/K_00_Business_Strategy.ipynb) | Synthese aller Kür-Analysen: Trigger-Matrix, Szenarien, Empfehlungen je Segment | ← Einstieg |
| [K_01 – Räumliche Analyse](../kuer/K_01_Raeumliche_Analyse.ipynb) | [BVI](../organisation/O_02_Glossar.ipynb#g-bvi), Zonenbilanzen, Standortoptimierung — wo lohnen sich Batterien räumlich? | |
| [K_02 – Cross-Border-Analyse](../kuer/K_02_Cross_Border.ipynb) | Grenzflüsse CH↔DE/AT/IT/FR — empirische Validierung des Business Case | |
| [K_03 – Marktdynamik](../kuer/K_03_Marktdynamik.ipynb) | Spread-Entwicklung 2018–2024, [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex)-Lernkurve bis 2035 | |
| [K_04 – Saisonale Animationen](../kuer/K_04_Animationen.ipynb) | Animierter Jahresverlauf: Spread, Netzlast, Arbitrage-Fenster als GIF | |
| [K_05 – Revenue Stacking](../kuer/K_05_Revenue_Stacking.ipynb) | [VPP](../organisation/O_02_Glossar.ipynb#g-vpp), FCR/aFRR, Smart Tariff — Erlösquellen neben reiner Arbitrage | |
| [K_06 – Dispatch-Optimierung](../kuer/K_06_Dispatch_Optimierung.ipynb) | Day-Ahead-optimaler [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) vs. reaktives Schwellenwertmodell | |
| [K_07 – Technologievergleich](../kuer/K_07_Technologievergleich.ipynb) | LFP, [NMC](../organisation/O_02_Glossar.ipynb#g-nmc), [NaS](../organisation/O_02_Glossar.ipynb#g-nas), [Redox-Flow](../organisation/O_02_Glossar.ipynb#g-redox-flow), Pumpspeicher: Profil und Segmenteignung | |
| [K_08 – Alternative Speicher](../kuer/K_08_Alternative_Speicher.ipynb) | Pumpspeicher, [CAES](../organisation/O_02_Glossar.ipynb#g-caes), [Power-to-X](../organisation/O_02_Glossar.ipynb#g-power-to-x) — andere Kostenstruktur, andere Regulierung | |
| [K_09 – Eigenverbrauchsoptimierung](../kuer/K_09_Eigenverbrauch.ipynb) | Netz-Eigenverbrauch als Alternative und Ergänzung zur [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) | |
| [K_10 – Produktsteckbrief](../kuer/K_10_Produkt_Steckbrief.ipynb) | Heimspeicher 7 kW / 4 kWh: konkreter Business Case Arbitrage + Eigenverbrauch | |
| [K_99 – Kombinierte Simulation](../kuer/K_99_Kombinierte_Simulation.ipynb) | Hybrid-Dispatch: Arbitrage + Eigenverbrauch — 4-Strategie-Vergleich | |


---
## Abschluss <a id='abschluss_NB_00'></a>

[↑ Inhaltsverzeichnis](#toc_NB_00)

Pflicht-Charts aus `output/charts/` auf Existenz prüfen.
Fehlende Charts: NB03 erneut ausführen.


In [ ]:
# ── Abschluss-Übersicht ──────────────────────────────────────────────────────
# os bereits in Setup-Zelle importiert

MODE       = CFG.get('mode', 'data')
DIR_RAW    = os.path.join('../data', 'raw')
DIR_PROC   = os.path.join('../data', 'processed')
DIR_INTER  = os.path.join('../data', 'intermediate', SZ_AKTIV)

print('=' * 60)
print(f'NB00 – Abschluss  |  Szenario: {SZ_AKTIV}')
print('=' * 60)

print('\nNotebooks (Pflicht):')
for nb_name in ['01_Daten_Laden.ipynb','02_Daten_Bereinigung.ipynb',
                '03_Daten_Analyse.ipynb','04_Visualisierungen.ipynb']:
    print(f'  {"✅" if os.path.exists(nb_name) else "❌"}  {nb_name}')

print(f'\nZwischenergebnisse ({DIR_INTER}):')
if os.path.exists(DIR_INTER):
    for f in sorted(os.listdir(DIR_INTER)):
        kb = os.path.getsize(os.path.join(DIR_INTER, f)) / 1024
        print(f'  ✅  {f:<45} {kb:>7.1f} KB')
else:
    print(f'  ❌  Ordner nicht vorhanden — NB02 ausführen')

print(f'\nCharts ({CHARTS_DIR}):')
if os.path.exists(CHARTS_DIR):
    pngs = sorted(f for f in os.listdir(CHARTS_DIR) if f.endswith('.png'))
    for f in pngs:
        print(f'  ✅  {f}')
    print(f'  {len(pngs)} Chart(s) vorhanden')
else:
    print(f'  ❌  Ordner nicht vorhanden — NB03 ausführen')

print(f'\n→ Weiter mit: K_00 Business Strategy (Kür)')


| [← NB04 Visualisierungen](04_Visualisierungen.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB01 Daten Laden →](01_Daten_Laden.ipynb) |
|:---|:---:|---:|